# 03 Final Model Training

비교 단계에서 선택한 설정으로 최종 모델을 학습하고, Drive에 체크포인트와 메타데이터를 저장합니다.

In [ ]:
!pip install -q ultralytics pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/sessac_project')
OUTPUT_DIR = Path('/content/drive/MyDrive/sessac_project_artifacts/final_training')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR / '3_models') not in sys.path:
    sys.path.append(str(PROJECT_DIR / '3_models'))

from colab_paths import resolve_project_paths
from behavior_modeling import (
    TCNClassifier,
    build_dataloaders,
    build_group_folds,
    discover_behavior_samples,
    fit_model,
    load_behavior_arrays,
    summarize_history,
)
from yolo_workflow import prepare_yolo_dataset, train_yolo_model, write_yolo_yaml

paths = resolve_project_paths(PROJECT_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# 행동 인식 최종 학습 설정
records = discover_behavior_samples(
    labels_root=paths['behavior_labels'],
    landmarks_root=paths['behavior_landmarks'],
    legacy_labels_root=paths['legacy_labels'],
    legacy_landmarks_root=paths['legacy_landmarks'],
)
data_dict = load_behavior_arrays(records)
folds = build_group_folds(records, num_folds=4)
final_fold = folds[0]

train_dataset, val_dataset, train_loader, val_loader = build_dataloaders(
    data_dict=data_dict,
    fold_info=final_fold,
    batch_size=64,
    window_size=15,
    step_size=5,
)
sample_batch = next(iter(train_loader))
input_dim = sample_batch['x'].shape[-1]
num_classes = sample_batch['y_last'].shape[-1]

behavior_model = TCNClassifier(input_dim=input_dim, num_classes=num_classes, channels=(64, 64)).to(device)
behavior_model, behavior_history = fit_model(
    behavior_model,
    train_loader,
    val_loader,
    device,
    epochs=80,
    patience=12,
)
behavior_summary = summarize_history(behavior_history)
behavior_output_dir = OUTPUT_DIR / 'behavior'
behavior_output_dir.mkdir(parents=True, exist_ok=True)
torch.save({
    'model_state_dict': behavior_model.state_dict(),
    'input_dim': input_dim,
    'num_classes': num_classes,
    'window_size': 15,
    'step_size': 5,
    'summary': behavior_summary,
    'train_keys': final_fold['train_keys'],
    'val_keys': final_fold['val_keys'],
}, behavior_output_dir / 'behavior_tcn_final.pt')
pd.DataFrame(behavior_history).to_csv(behavior_output_dir / 'behavior_tcn_history.csv', index=False, encoding='utf-8-sig')
(behavior_output_dir / 'behavior_tcn_summary.json').write_text(json.dumps(behavior_summary, indent=2, ensure_ascii=False), encoding='utf-8')
behavior_summary

In [ ]:
# YOLO 최종 학습 설정
yolo_output_dir = OUTPUT_DIR / 'yolo'
yolo_output_dir.mkdir(parents=True, exist_ok=True)

open_close_dataset = prepare_yolo_dataset(
    image_root=paths['yolo_images'],
    label_root=paths['yolo_labels_open_close'],
    output_root=yolo_output_dir / 'dataset_open_close',
    val_ratio=0.2,
    seed=42,
)
open_close_yaml = write_yolo_yaml(
    dataset_root=open_close_dataset['dataset_root'],
    yaml_path=yolo_output_dir / 'open_close.yaml',
    class_names=['open', 'close'],
)
open_close_result = train_yolo_model(
    yaml_path=open_close_yaml,
    model_name='yolov8s.pt',
    project_dir=yolo_output_dir / 'runs',
    run_name='open_close_final',
    epochs=50,
    image_size=640,
    batch_size=16,
    device=0,
)

full_empty_dataset = prepare_yolo_dataset(
    image_root=paths['yolo_images'],
    label_root=paths['yolo_labels_full_empty'],
    output_root=yolo_output_dir / 'dataset_full_empty',
    val_ratio=0.2,
    seed=42,
)
full_empty_yaml = write_yolo_yaml(
    dataset_root=full_empty_dataset['dataset_root'],
    yaml_path=yolo_output_dir / 'full_empty.yaml',
    class_names=['empty', 'full'],
)
full_empty_result = train_yolo_model(
    yaml_path=full_empty_yaml,
    model_name='yolov8s.pt',
    project_dir=yolo_output_dir / 'runs',
    run_name='full_empty_final',
    epochs=50,
    image_size=640,
    batch_size=16,
    device=0,
)

final_summary = {
    'behavior': behavior_summary,
    'yolo_open_close': open_close_result,
    'yolo_full_empty': full_empty_result,
}
(OUTPUT_DIR / 'final_summary.json').write_text(json.dumps(final_summary, indent=2, ensure_ascii=False), encoding='utf-8')
final_summary